# F2 · Preprocesamiento de licitaciones de ChileCompra

**Pregunta descriptiva propuesta.** ¿Cómo se distribuye la diferencia entre el monto estimado registrado y la suma de las ofertas ganadoras por licitación, por moneda y visibilidad del estimado? ¿En qué proporción coinciden ambos montos?

La brecha describe diferencias entre valores registrados. Para interpretarla como desviación de un presupuesto previo haría falta verificar que el estimado se fijó antes de conocer la adjudicación. La base de ofertas se mantiene a nivel de oferta por ítem. La brecha se guarda en una tabla aparte, con una fila por licitación.

## 0. Verificar el entorno de ejecución

Antes de cargar datos comprobamos que el notebook se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías que usa esta fase tienen las versiones declaradas en `requirements.txt`. Es la misma verificación de la sección 8 de `F1_Definicion`, reducida a lo que necesita la Fase 2.

La celda se detiene si no hay entorno virtual o si falta una librería, y avisa si alguna versión no coincide con la declarada.

In [1]:
# Verificar el entorno: intérprete, entorno virtual y versiones frente a requirements.txt.
import platform
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

# Paquetes que usa la Fase 2, con el nombre que tienen en requirements.txt.
PAQUETES_F2 = ["pandas", "numpy"]


def leer_versiones_declaradas(ruta_requirements, paquetes):
    """Devuelve {paquete: versión} de requirements.txt para los paquetes pedidos."""
    declaradas = {}
    for linea in ruta_requirements.read_text(encoding="utf-8").splitlines():
        nombre, _, version = linea.strip().partition("==")
        if nombre.lower() in paquetes:
            declaradas[nombre.lower()] = version
    return declaradas


def verificar_entorno(paquetes):
    """Comprueba el entorno virtual y compara versiones instaladas y declaradas.

    Se detiene si no hay entorno virtual o si falta un paquete. Devuelve una
    tabla con la versión declarada e instalada de cada paquete.
    """
    inicio = Path.cwd().resolve()
    raiz = next((p for p in (inicio, *inicio.parents) if (p / "requirements.txt").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró requirements.txt: abre el notebook dentro del repositorio.")
    if sys.prefix == sys.base_prefix:
        raise RuntimeError("No se usa un entorno virtual: cambia el kernel a 'Python (grupo2-mcdi500)'.")

    declaradas = leer_versiones_declaradas(raiz / "requirements.txt", paquetes)
    filas = []
    for paquete in paquetes:
        try:
            instalada = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            instalada = None
        filas.append({"Paquete": paquete, "Declarada": declaradas.get(paquete),
                      "Instalada": instalada, "Coincide": instalada == declaradas.get(paquete)})
    tabla = pd.DataFrame(filas)

    faltantes = tabla.loc[tabla["Instalada"].isna(), "Paquete"].tolist()
    if faltantes:
        raise ModuleNotFoundError(f"Faltan paquetes: {faltantes}. Ejecuta: pip install -r requirements.txt")

    interprete = Path(sys.executable)
    if interprete.is_relative_to(raiz):
        interprete = interprete.relative_to(raiz)
    print(f"Intérprete: {interprete} | Python {platform.python_version()} | {platform.system()}")
    print("[OK] Entorno virtual activo")
    return tabla


versiones = verificar_entorno(PAQUETES_F2)
display(versiones)
if not versiones["Coincide"].all():
    print("[AVISO] Hay versiones distintas a las declaradas: revisa requirements.txt.")
else:
    print(f"[OK] Las {len(versiones)} versiones coinciden con requirements.txt")

Intérprete: .venv/bin/python | Python 3.12.3 | Darwin
[OK] Entorno virtual activo


,Paquete,Declarada,Instalada,Coincide
0,pandas,3.1.0,3.1.0,True
1,numpy,2.5.4,2.5.4,True


[OK] Las 2 versiones coinciden con requirements.txt


## 1. Configurar el proyecto

El archivo de entrada no se versiona. Descárgalo y déjalo en `data/raw/04LicitacionesEne.csv` siguiendo la sección *Obtener los datos* del `README.md` (detalle y verificación SHA-256 en `data/raw/README.md`).

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

actual = Path.cwd()
raiz = next((
    carpeta for carpeta in (actual, *actual.parents)
    if (carpeta / "src/funciones_f2_revision.py").is_file()
), None)
if raiz is None:
    raise FileNotFoundError("Ubica funciones_f2_revision.py y pruebas_f2.py en src/ del proyecto.")
sys.path.insert(0, str(raiz))
from src.funciones_f2_revision import (
    MOTIVOS, aplicar_bandera_atipicos, calcular_brecha, calcular_limites_iqr,
    cargar_csv, convertir_tipos, marcar_duplicados, normalizar_codigo,
    ordenar_tamano, perfilar_columnas, tratar_faltantes, validar_procesado,
)
from src.pruebas_f2 import ejecutar_pruebas_f2

ruta = raiz / "data/raw/04LicitacionesEne.csv"
print("Entrada:", ruta.relative_to(raiz))
print("Versión de pandas:", pd.__version__)

Entrada: data/raw/04LicitacionesEne.csv
Versión de pandas: 3.1.0


## 2. Verificar las reglas con datos ficticios

In [3]:
ejecutar_pruebas_f2()
print("Pruebas de brecha, duplicados, tamaño, esquema e IQR aprobadas.")

Pruebas de brecha, duplicados, tamaño, esquema e IQR aprobadas.


## 3. Cargar, perfilar y revisar duplicados

Se excluyen los identificadores directos detectados en los encabezados al leer. Se marcan las coincidencias exactas de las columnas restantes **antes de transformar tipos o recortar columnas**. Tras excluir identificadores, dos filas pueden verse iguales y corresponder a ofertas legítimas. Se conservan las filas y se reporta cuántas coincidencias involucran ofertas ganadoras. Se compara después la brecha con y sin licitaciones afectadas para medir sensibilidad sin eliminar ofertas automáticamente.

In [4]:
datos = cargar_csv(ruta)
filas_originales = len(datos)
print(f"Entrada: {filas_originales:,} filas y {datos.shape[1]} columnas")
perfil = perfilar_columnas(datos)
display(perfil)
nulos_entrada = datos.isna().sum()
datos, resumen_duplicados = marcar_duplicados(datos)
print("Filas adicionales idénticas por resultado:")
display(resumen_duplicados)

Entrada: 45,741 filas y 71 columnas


,Tipo,Nulos,% nulos,Valores distintos
NroLicitacion,string,0,0.0,2292
NombreLicitacion,str,0,0.0,2282
TipoLicitacion,str,0,0.0,9
Descripcion,str,0,0.0,2222
MonedaLicitacion,str,0,0.0,4
...,...,...,...,...
CantidadOferta,int64,0,0.0,358
MonedaOferta,str,0,0.0,4
MontoNetoOferta,str,0,0.0,16765
MontoTotalOferta,str,0,0.0,21376


Filas adicionales idénticas por resultado:


,FilasEnGrupoIdentico,FilasAdicionales
ResultadoOferta,,
Ganadora,22,11
Perdedora,22,11


## 4. Convertir y decidir qué hacer con los faltantes

Se separan las ausencias originales de los fallos de conversión. Para cada variable del análisis se presenta su regla y motivo. Las columnas con más del 90 % de faltantes se revisan por separado. La exportación de ofertas usa una lista explícita de variables analíticas y excluye campos de texto libre como nombres de ofertas y descripciones.

In [5]:
datos, conversiones = convertir_tipos(datos)
display(conversiones)
datos, faltantes = tratar_faltantes(datos, nulos_entrada.reindex(datos.columns))
faltantes = faltantes.drop(index="EsDuplicadoPotencial")
display(faltantes.loc[faltantes.index.intersection(MOTIVOS)])
display(faltantes.loc[faltantes["% faltantes"].gt(90)].sort_values("% faltantes", ascending=False))
datos = ordenar_tamano(datos)
display(datos["TamanoProveedorOrdinal"].value_counts(dropna=False).sort_index())

,Nulos en entrada,Fallos de conversión
Columna,,
FechaPublicacion,0,0
FechaCierre,0,0
FechaAdjudicacion,91,0
MontoNetoOferta,0,0
MontoTotalOferta,0,0
MontoEstimadoLicitacion,0,0


,Nulos al leer,Nulos después de convertir y normalizar,% faltantes,Regla,Motivo
NroLicitacion,0.0,0,0.000000,Conservar faltantes,"Identifica la licitación. Sin código, la fila ..."
MonedaLicitacion,0.0,0,0.000000,Conservar faltantes,"Unidad del estimado. Si falta, no puede verifi..."
MontoEstimadoLicitacion,0.0,0,0.000000,Conservar faltantes,Referencia de comparación. Sin monto válido se...
MontoEstimadoVisible,0.0,0,0.000000,Conservar faltantes,Distingue visibilidad pública del monto; vacío...
FechaPublicacion,0.0,0,0.000000,Conservar faltantes,Permite describir cuándo se publicó. Se descon...
FechaCierre,0.0,0,0.000000,Conservar faltantes,Permite describir el plazo de recepción cuando...
FechaAdjudicacion,91.0,91,0.198946,Conservar faltantes,Permite estudiar fechas de adjudicación observ...
CodigoProductoONU,0.0,0,0.000000,Conservar faltantes,Permite segmentar por producto cuando hay código.
TamanoProveedor,0.0,0,0.000000,Conservar faltantes,Describe tamaño. Una ausencia queda sin rango ...
CantidadOferta,0.0,0,0.000000,Conservar faltantes,Permite contrastar el cálculo del monto total ...


,Nulos al leer,Nulos después de convertir y normalizar,% faltantes,Regla,Motivo
LicitacionBaseTipo,45741.0,45741,100.000000,Excluir del CSV analítico,Sin observaciones y fuera del análisis.
ContratoRenovable,45741.0,45741,100.000000,Excluir del CSV analítico,Sin observaciones y fuera del análisis.
UnidadTiempoRenovacion,45741.0,45741,100.000000,Excluir del CSV analítico,Sin observaciones y fuera del análisis.
FechaEntregaEnSoporteFisico,45457.0,45457,99.379113,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
FechaEstimadaFirmaContrato,45270.0,45270,98.970289,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
RazonPublicidadOfertasTecnicas,44018.0,44018,96.233139,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
FechaEstimadaEvaluacionOfertas,43489.0,43489,95.076627,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...


TamanoProveedorOrdinal
Micro       6539
Pequeña    13510
Mediana     8570
Grande     11848
NaN         5274
Name: count, dtype: int64

## 5. Calcular la brecha por licitación

Se suman los montos totales de **todas las ofertas ganadoras** por licitación. El estimado debe ser positivo, estar presente y ser consistente en las filas de la licitación. Se requiere una moneda de licitación consistente y que cada ganadora tenga la misma moneda y un monto positivo. También se excluyen licitaciones con resultados de oferta faltantes para evitar sumas parciales.

**Brecha = suma de montos ganadores − monto estimado registrado.** Se reporta la cobertura por licitación y se resume por moneda. Para identificar coincidencias se usa una tolerancia absoluta de 0,00000001 por el redondeo numérico. La comparación de sensibilidad excluye licitaciones con coincidencias exactas entre filas ganadoras.

In [6]:
licitaciones = calcular_brecha(datos)
comparables = licitaciones.loc[licitaciones["EstadoBrecha"].eq("Calculada")]
sin_duplicadas = comparables.loc[~comparables["GanadoraDuplicada"]]

print(f"Licitaciones con código: {len(licitaciones):,}")
print(f"Filas sin código: {normalizar_codigo(datos['NroLicitacion']).isna().sum():,}")
display(licitaciones["EstadoBrecha"].value_counts().to_frame("Licitaciones"))
print(f"Brechas calculadas: {len(comparables):,} de {len(licitaciones):,}")
if len(licitaciones):
    print(f"Cobertura: {len(comparables) / len(licitaciones):.1%}")
for nombre, muestra in [("Todas las comparables", comparables), ("Sin ganadoras duplicadas potenciales", sin_duplicadas)]:
    print(nombre)
    display(muestra.groupby("MonedaLicitacion").agg(
        Licitaciones=("BrechaMonto", "size"),
        Coinciden=("CoincideConEstimado", "sum"),
        ProporcionCoincide=("CoincideConEstimado", "mean"),
        MedianaBrecha=("BrechaMonto", "median"),
    ))

Licitaciones con código: 2,292
Filas sin código: 0


,Licitaciones
EstadoBrecha,
Calculada,2257
Sin ofertas ganadoras,26
Estimado inválido o inconsistente,8
Ganadora con monto inválido,1


Brechas calculadas: 2,257 de 2,292
Cobertura: 98.5%
Todas las comparables


,Licitaciones,Coinciden,ProporcionCoincide,MedianaBrecha
MonedaLicitacion,,,,
CLF,20,0,0.0,-60.36
CLP,2230,1252,0.561435,0.0
USD,3,2,0.666667,0.0
UTM,4,0,0.0,-31.64


Sin ganadoras duplicadas potenciales


,Licitaciones,Coinciden,ProporcionCoincide,MedianaBrecha
MonedaLicitacion,,,,
CLF,20,0,0.0,-60.36
CLP,2226,1252,0.562444,0.0
USD,3,2,0.666667,0.0
UTM,4,0,0.0,-31.64


## 6. Revisar si el estimado sirve como referencia previa

El [diccionario de licitaciones de la API de Mercado Público](https://www.chilecompra.cl/wp-content/uploads/2026/03/Documentacion-API-Mercado-Publico-Licitaciones.pdf) describe el monto estimado de la institución y un indicador que determina si se publica en la ficha. Es documentación de la API. No establece cuándo se registró el valor en este extracto CSV ni confirma su independencia de lo adjudicado.

Se contrasta la coincidencia por moneda y visibilidad. Los campos `BaseEstimacionMontoLicitacion` y `JustificacionMontoEstimado` se examinan mediante conteos de presencia y menciones a adjudicación o presupuesto. Estas menciones orientan la revisión y requieren interpretación del equipo. Los textos se quedan en el CSV local y no se muestran ni se exportan.

In [7]:
if "MontoEstimadoVisible" in datos:
    visible = normalizar_codigo(datos["MontoEstimadoVisible"]).str.replace("Í", "I")
    print("Filas de oferta con monto indicado como no visible:", visible.eq("NO").sum())
    print("Proporción del total de filas:", round(visible.eq("NO").sum() / len(datos), 4) if len(datos) else 0)
    display(comparables.groupby(["MonedaLicitacion", "VisibilidadEstimado"]).agg(
        Licitaciones=("BrechaMonto", "size"),
        Coinciden=("CoincideConEstimado", "sum"),
        ProporcionCoincide=("CoincideConEstimado", "mean"),
    ))

campos_contexto = [c for c in licitaciones if c.endswith((
    "_Informada", "_MencionaAdjudicacion", "_MencionaPresupuesto",
))]
if campos_contexto:
    display(licitaciones[campos_contexto].sum().rename("Licitaciones").to_frame())
    display(comparables.groupby("CoincideConEstimado")[campos_contexto].sum())

Filas de oferta con monto indicado como no visible: 24895
Proporción del total de filas: 0.5443


Licitaciones  Coinciden  \
MonedaLicitacion VisibilidadEstimado                            
CLF              NO                             10          0   
                 SI                             10          0   
CLP              NO                           1189        635   
                 SI                           1041        617   
USD              NO                              1          1   
                 SI                              2          1   
UTM              NO                              4          0   

                                      ProporcionCoincide  
MonedaLicitacion VisibilidadEstimado                      
CLF              NO                                  0.0  
                 SI                                  0.0  
CLP              NO                             0.534062  
                 SI                             0.592699  
USD              NO                                  1.0  
                 SI                                  0.5  
UTM              NO                                  0.0

,Licitaciones
BaseEstimacionMontoLicitacion_Informada,2292
BaseEstimacionMontoLicitacion_MencionaAdjudicacion,0
BaseEstimacionMontoLicitacion_MencionaPresupuesto,1603
JustificacionMontoEstimado_Informada,800
JustificacionMontoEstimado_MencionaAdjudicacion,3
JustificacionMontoEstimado_MencionaPresupuesto,264


,BaseEstimacionMontoLicitacion_Informada,BaseEstimacionMontoLicitacion_MencionaAdjudicacion,BaseEstimacionMontoLicitacion_MencionaPresupuesto,JustificacionMontoEstimado_Informada,JustificacionMontoEstimado_MencionaAdjudicacion,JustificacionMontoEstimado_MencionaPresupuesto
CoincideConEstimado,,,,,,
False,1003,0,666,313,2,105
True,1254,0,907,475,1,155


## 7. Marcar valores extremos en CLP con IQR sobre logaritmos

Se aplica IQR al logaritmo natural de los montos positivos en CLP. El logaritmo permite trabajar con una distribución de montos asimétrica. Se reportan los límites en la escala original y cuántos casos se marcan en cada extremo. La bandera indica un **valor extremo** que requiere contexto. Se conservan todas las filas.

In [8]:
limites = calcular_limites_iqr(datos)
datos = aplicar_bandera_atipicos(datos, limites)
if limites is not None:
    inferior = np.exp(limites["inferior_log"])
    superior = np.exp(limites["superior_log"])
    evaluadas = datos["EsExtremoLogIQR_CLP"].notna()
    montos = datos.loc[evaluadas, "MontoTotalOferta"]
    display(pd.DataFrame([{
        **limites, "Límite inferior CLP": inferior, "Límite superior CLP": superior,
        "Extremos inferiores": int(np.log(montos).lt(limites["inferior_log"]).sum()),
        "Extremos superiores": int(np.log(montos).gt(limites["superior_log"]).sum()),
    }]))
display(datos["EsExtremoLogIQR_CLP"].value_counts(dropna=False).to_frame("Ofertas"))

,ofertas_validas,q1_log,q3_log,inferior_log,superior_log,Límite inferior CLP,Límite superior CLP,Extremos inferiores,Extremos superiores
0,45603,10.683179,14.404324,5.101462,19.986041,164.261932,4.784398e+08,810,77


,Ofertas
EsExtremoLogIQR_CLP,
False,44716
True,887
<NA>,138


## 8. Validar y exportar

Se comprueban el número de filas, la cardinalidad de licitaciones, la escala ordinal, la cobertura de la brecha y de los extremos, y las columnas permitidas. El CSV de ofertas conserva el nombre de salida anterior. La tabla de licitaciones contiene la nueva brecha. Para que F3 reproduzca F2, deberá trabajar con esas dos unidades y conservar los parámetros IQR del ajuste.

In [9]:
columnas_ofertas = [c for c in MOTIVOS if c in datos]
columnas_ofertas += ["EsDuplicadoPotencial", "TamanoProveedorOrdinal", "EsExtremoLogIQR_CLP"]
ofertas = datos[columnas_ofertas]
validar_procesado(ofertas, licitaciones, filas_originales, limites)

salida = raiz / "data/processed"
salida.mkdir(parents=True, exist_ok=True)
for tabla, nombre in [
    (ofertas, "04LicitacionesEne_procesado.csv"),
    (licitaciones, "04LicitacionesEne_brecha_por_licitacion.csv"),
]:
    destino = salida / nombre
    temporal = destino.with_suffix(".csv.tmp")
    tabla.to_csv(temporal, sep=";", index=False, encoding="utf-8-sig")
    temporal.replace(destino)
    print(f"{destino.relative_to(raiz)}: {len(tabla):,} filas × {tabla.shape[1]} columnas")

data/processed/04LicitacionesEne_procesado.csv: 45,741 filas × 17 columnas
data/processed/04LicitacionesEne_brecha_por_licitacion.csv: 2,292 filas × 24 columnas
